In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import faiss
import json
import random
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

print(f"NumPy version: {np.__version__}")
print(f"FAISS version: {faiss.__version__}")

NumPy version: 2.3.5
FAISS version: 1.13.2


In [15]:
knowledge_base = {
    "doc_001": {
        "title": "Основы здорового питания",
        "content": """Здоровое питание - это основа хорошего самочувствия и долголетия.
        Оно включает баланс белков, жиров и углеводов в соотношении примерно 30/20/50.
        Рекомендуется потреблять 50-55% углеводов, 15-20% белков и 25-30% жиров от общей калорийности.
        Важно есть разнообразную пищу: овощи, фрукты, цельнозерновые продукты, нежирное мясо, рыбу, бобовые.
        Также необходимо ограничивать потребление соли (не более 5 г в день), сахара (не более 50 г) и насыщенных жиров.
        Питьевой режим играет важную роль - нужно выпивать не менее 1.5-2 литров воды в день.
        Регулярное питание небольшими порциями 4-5 раз в день помогает поддерживать стабильный уровень сахара в крови.
        Исследования показывают, что средиземноморская диета снижает риск сердечно-сосудистых заболеваний на 30%."""
    },

    "doc_002": {
        "title": "Польза белков для организма",
        "content": """Белки являются строительным материалом для мышц, костей, кожи, волос, ногтей и ферментов.
        Они состоят из аминокислот, 9 из которых являются незаменимыми (организм не может их синтезировать).
        Суточная норма белка для взрослого человека с нормальной активностью - 0.8-1 грамм на килограмм веса.
        При активных тренировках норма увеличивается до 1.6-2.2 г/кг веса. Для пожилых людей рекомендуется 1.2-1.5 г/кг.
        Источники животного белка: курица, индейка, говядина, рыба, яйца, молочные продукты, творог, сыр.
        Растительные источники: бобовые (чечевица, нут, фасоль), тофу, темпе, сейтан, киноа, гречка, орехи, семена.
        Растительные белки менее полноценны из-за недостатка некоторых аминокислот, поэтому их лучше комбинировать.
        Например, рис с бобами или хумус с цельнозерновым хлебом дают полный набор аминокислот.
        Переизбыток белка может нагружать почки и печень, а также приводить к обезвоживанию."""
    },

    "doc_003": {
        "title": "Здоровые жиры и их роль",
        "content": """Жиры необходимы для усвоения жирорастворимых витаминов (A, D, E, K), производства гормонов и здоровья клеточных мембран.
        Ненасыщенные жиры полезны для сердца и сосудов, они снижают уровень "плохого" холестерина.
        Мононенасыщенные жиры содержатся в оливковом масле, авокадо, орехах (миндаль, фундук), семенах кунжута.
        Полиненасыщенные жиры (Омега-3 и Омега-6) есть в жирной рыбе (лосось, скумбрия, сардины), грецких орехах, семенах льна и чиа.
        Насыщенные жиры следует ограничивать до 10% калорийности: они есть в сливочном масле, сыре, жирном мясе, кокосовом и пальмовом масле.
        Трансжиры полностью вредны и содержатся в фастфуде, промышленной выпечке, маргарине, магазинных соусах.
        Омега-3 жирные кислоты особенно важны для мозга, снижения воспалений и профилактики депрессии.
        Рекомендуется употреблять жирную рыбу не менее 2-3 раз в неделю или принимать рыбий жир.
        Авокадо и оливковое масло первого отжима считаются суперфудами для сердечно-сосудистой системы."""
    },

    "doc_004": {
        "title": "Углеводы: простые и сложные",
        "content": """Углеводы - основной источник энергии для организма, особенно для мозга и мышц.
        Сложные углеводы (цельнозерновые продукты, овощи, бобовые) дают энергию постепенно и богаты клетчаткой, витаминами группы B.
        Они имеют низкий гликемический индекс, что помогает контролировать аппетит и уровень сахара в крови.
        Примеры сложных углеводов: овсянка, гречка, бурый рис, киноа, цельнозерновой хлеб, сладкий картофель, бобовые.
        Простые углеводы (сахар, сладости, белый хлеб, белый рис, сладкие напитки) быстро повышают сахар крови.
        Они дают быстрый прилив энергии, но затем следует резкий спад, вызывая усталость и голод.
        Рекомендуется отдавать предпочтение сложным углеводам и ограничивать добавленный сахар до 25-50 г в день.
        Клетчатка (20-35 г в день) важна для пищеварения, контроля веса и снижения холестерина.
        Источники клетчатки: овощи, фрукты, ягоды, отруби, цельнозерновые продукты, семена чиа и льна."""
    },

    "doc_005": {
        "title": "Режим питания и контроль порций",
        "content": """Правильный режим питания помогает контролировать вес и улучшает метаболизм.
        Рекомендуется есть 3-5 раз в день небольшими порциями через каждые 3-4 часа.
        Завтрак должен быть плотным и составлять 25-30% калорийности дня, лучше в течение часа после пробуждения.
        Обед - 35-40% калорий, самый сытный прием пищи. Ужин - 20-25%, легкий, за 2-3 часа до сна.
        Перекусы (2-3 в день) - 10-15% калорий, предпочтительны фрукты, орехи, йогурт.
        Контроль порций: тарелка должна быть наполовину заполнена овощами, на четверть - белком, на четверть - сложными углеводами.
        Размер порции белка - с ладонь, углеводов - с кулак, жиров - с большой палец.
        Важно пить достаточно воды: 30 мл на килограмм веса в день, равномерно в течение дня.
        Не рекомендуется пропускать приемы пищи - это замедляет метаболизм и ведет к перееданию вечером."""
    },

    "doc_006": {
        "title": "Витамин D: солнечный витамин",
        "content": """Витамин D критически важен для здоровья костей, иммунитета, настроения и профилактики многих заболеваний.
        Он синтезируется в коже под воздействием солнечного света (UVB-лучи).
        Дефицит витамина D наблюдается у 80% населения России, особенно в зимний период.
        Симптомы дефицита: усталость, мышечная слабость, частые простуды, депрессия, боли в костях.
        Суточная норма: 600-800 МЕ для взрослых, 800-1000 МЕ для пожилых и 400-800 МЕ для детей.
        При дефиците врач может назначить лечебные дозы до 2000-5000 МЕ в день.
        Продукты, богатые витамином D: жирная рыба (лосось, сельдь, скумбрия), печень трески, яичные желтки, грибы шиитаке.
        В зимний период рекомендуется принимать добавки витамина D, особенно в регионах севернее 45° широты.
        Витамин D лучше усваивается с жирной пищей, так как он жирорастворимый."""
    },

    "doc_007": {
        "title": "Витамин C и иммунитет",
        "content": """Витамин C (аскорбиновая кислота) - мощный антиоксидант, укрепляющий иммунитет и защищающий клетки от повреждений.
        Он участвует в синтезе коллагена - белка, необходимого для здоровья кожи, суставов, костей и кровеносных сосудов.
        Суточная норма: 75-90 мг для взрослых, при беременности и кормлении - до 120 мг, курильщикам нужно на 35 мг больше.
        Витамин C улучшает усвоение железа из растительной пищи, особенно в сочетании с продуктами, богатыми железом.
        Лучшие источники: шиповник, черная смородина, киви, болгарский перец (красный и зеленый), цитрусовые, брокколи, брюссельская капуста.
        Термическая обработка разрушает до 50-70% витамина C, поэтому овощи и фрукты лучше есть свежими или готовить на пару.
        При простуде высокие дозы витамина C (до 2000 мг) могут сократить продолжительность болезни на 1-2 дня.
        Переизбыток витамина C выводится с мочой, но очень высокие дозы могут вызвать диарею и проблемы с почками."""
    },

    "doc_008": {
        "title": "Кальций для костей и зубов",
        "content": """Кальций - основной минерал для костей, зубов, нормальной работы сердца, мышц и нервной системы.
        99% кальция в организме находится в костях и зубах, остальной 1% - в крови и тканях.
        Суточная норма: 1000 мг для взрослых 19-50 лет, 1200 мг для женщин после 50 и мужчин после 70 лет.
        При дефиците кальция организм начинает "забирать" его из костей, что приводит к остеопорозу и повышенному риску переломов.
        Лучшие источники: молочные продукты (сыр, творог, йогурт, молоко - 300 мг в стакане), кунжут (980 мг в 100 г), сардины с костями.
        Растительные источники: зеленые листовые овощи (кале, брокколи), миндаль, тофу, обогащенное растительное молоко, бобовые.
        Для усвоения кальция необходим витамин D, а также магний и витамин K2.
        Кофеин, избыток соли и фосфорной кислоты (газировка) мешают усвоению кальция."""
    },

    "doc_009": {
        "title": "Железо и профилактика анемии",
        "content": """Железо необходимо для производства гемоглобина - белка в красных кровяных клетках, переносящего кислород.
        Железодефицитная анемия - самое распространенное дефицитное состояние, особенно у женщин детородного возраста.
        Симптомы анемии: усталость, бледность, одышка, головокружение, ломкость ногтей и волос, холодные руки и ноги.
        Суточная норма: 8-11 мг для мужчин, 15-18 мг для женщин, 27 мг во время беременности.
        Гемовое железо (из животных продуктов) усваивается лучше всего: красное мясо, печень, почки, птица, рыба, морепродукты.
        Негемовое железо (из растений) усваивается хуже: гречка, шпинат, бобовые, тофу, кунжут, тыквенные семечки, сухофрукты.
        Усвоение негемового железа улучшает витамин C (цитрусовые, болгарский перец, киви).
        Чай, кофе, молочные продукты и фитаты (в зерновых и бобовых) мешают усвоению железа. Лучше употреблять их отдельно от железосодержащих продуктов."""
    },

    "doc_010": {
        "title": "Магний: антистрессовый минерал",
        "content": """Магний участвует в более чем 300 биохимических реакциях: производство энергии, синтез белка, работа мышц и нервов, контроль сахара крови.
        Он помогает бороться со стрессом, улучшает сон, снижает тревожность и мышечные спазмы.
        Симптомы дефицита: мышечные судороги, усталость, бессонница, тревожность, аритмия, мигрени.
        Суточная норма: 400-420 мг для мужчин, 310-320 мг для женщин, при беременности - до 350-400 мг.
        Лучшие источники: тыквенные семечки (150 мг в 30 г), миндаль, кешью, грецкие орехи, шпинат, авокадо, бананы.
        Также магний есть в темном шоколаде (от 70% какао), бобовых, цельнозерновых продуктах, морской капусте.
        Алкоголь, кофеин, диуретики и стресс увеличивают потерю магния организмом.
        Цитрат и глицинат магния лучше усваиваются, чем оксид магния."""
    },

    "doc_011": {
        "title": "Рекомендации ВОЗ по физической активности",
        "content": """Всемирная организация здравоохранения (ВОЗ) дает четкие рекомендации по физической активности для разных возрастных групп.
        Взрослые люди 18-64 лет должны заниматься умеренной аэробной активностью 150-300 минут в неделю.
        Альтернатива - интенсивная аэробная активность 75-150 минут в неделю.
        Также нужны силовые тренировки всех основных групп мышц 2 и более раз в неделю.
        Умеренная активность: быстрая ходьба (5-6 км/ч), танцы, садоводство, домашние дела, теннис.
        Интенсивная активность: бег, плавание, велоспорт (более 15 км/ч), аэробика, футбол, баскетбол.
        Дети и подростки 5-17 лет нуждаются в 60 минутах умеренной или интенсивной активности ежедневно.
        Пожилые люди (65+ лет) дополнительно должны включать упражнения на равновесие для предотвращения падений.
        Регулярная активность снижает риск сердечно-сосудистых заболеваний, диабета 2 типа, рака груди и толстой кишки на 20-30%."""
    },

    "doc_012": {
        "title": "Кардиотренировки: польза и виды",
        "content": """Кардио упражнения укрепляют сердечно-сосудистую систему, улучшают кровообращение и повышают выносливость.
        Они сжигают много калорий, помогают снизить вес и уменьшить риск сердечных заболеваний, гипертонии, диабета.
        Кардио также снижает стресс, улучшает настроение за счет выработки эндорфинов и нормализует сон.
        Основные виды кардио: бег, ходьба, плавание, велоспорт, гребля, прыжки со скакалкой, эллиптический тренажер.
        Интервальные тренировки (чередование высокой и низкой интенсивности) наиболее эффективны для жиросжигания.
        Оптимальная продолжительность кардио для здоровья - 30-60 минут 3-5 раз в неделю.
        Новичкам начинать с 15-20 минут, постепенно увеличивая время и интенсивность.
        Зоны пульса: жиросжигание - 60-70% от максимального пульса (220 минус возраст), кардио - 70-80%, анаэробная - 80-90%.
        Лучшее время для кардио - утром натощак или через 1-2 часа после еды."""
    },

    "doc_013": {
        "title": "Силовые тренировки: база для здоровья",
        "content": """Силовые тренировки необходимы для наращивания и поддержания мышечной массы, укрепления костей и суставов.
        Они ускоряют метаболизм (мышцы сжигают больше калорий даже в покое), улучшают осанку и повседневную активность.
        Базовые упражнения, задействующие несколько групп мышц: приседания, отжимания, подтягивания, жим лежа, становая тяга, выпады.
        Новичкам рекомендуется выполнять 8-12 повторений в подходе, 2-3 подхода на упражнение, отдых 60-90 секунд.
        Тренировки должны охватывать все основные группы мышц: грудь, спину, плечи, ноги, ягодицы, пресс, руки.
        Прогрессия нагрузки: увеличивать вес, когда можете сделать 12 повторений с хорошей техникой.
        Техника важнее веса - неправильное выполнение может привести к травмам. Лучше начать с тренером.
        Силовые тренировки рекомендуется проводить 2-3 раза в неделю с отдыхом между тренировками 48 часов для восстановления мышц.
        Для похудения эффективны круговые тренировки (упражнения одно за другим без отдыха)."""
    },

    "doc_014": {
        "title": "Растяжка и гибкость: профилактика травм",
        "content": """Регулярная растяжка улучшает гибкость, подвижность суставов, снижает риск травм и уменьшает мышечное напряжение после тренировок.
        Она также улучшает кровообращение, осанку и помогает расслабиться, снижая уровень стресса.
        Статическая растяжка: удерживание положения в конечной точке движения в течение 15-30 секунд. Лучше всего после тренировки.
        Динамическая растяжка: контролируемые движения через полную амплитуду (махи ногами, вращения руками). Делать перед тренировкой для разогрева.
        Баллистическая растяжка (рывками) не рекомендуется из-за высокого риска травм.
        Проприоцептивная нейромышечная фасилитация (ПНФ) - продвинутая техника с партнером для максимальной растяжки.
        Йога и пилатес отлично развивают гибкость, силу и баланс одновременно.
        Растягиваться нужно до легкого дискомфорта, но не до боли. Дыхание должно быть ровным и глубоким.
        Регулярность важнее интенсивности: 10-15 минут растяжки ежедневно дадут больше результата, чем час раз в неделю."""
    },

    "doc_015": {
        "title": "Восстановление после тренировок",
        "content": """Восстановление - ключевой компонент прогресса в фитнесе, не менее важный, чем сами тренировки.
        Во время отдыха мышцы восстанавливаются и растут, восполняются запасы гликогена, нормализуется нервная система.
        Активное восстановление: легкая активность (ходьба, плавание, йога) на следующий день после интенсивной тренировки.
        Сон критически важен: во время глубокого сна вырабатывается гормон роста, происходит ремонт тканей.
        Питание после тренировки: углеводы (восполнить энергию) + белок (для восстановления мышц) в течение 30-60 минут после нагрузки.
        Гидратация: пить воду до, во время и после тренировки, при интенсивных нагрузках - изотонические напитки.
        Массаж, пенный ролл (самомассаж валиком) и компрессионная одежда улучшают кровообращение и уменьшают мышечную боль.
        Признаки недостаточного восстановления: постоянная усталость, падение производительности, раздражительность, частые болезни, бессонница.
        Рекомендуется делать 1-2 полных дня отдыха от тренировок в неделю и разгрузочную неделю каждые 6-8 недель."""
    },

    "doc_016": {
        "title": "Вода и гидратация организма",
        "content": """Вода составляет 60-70% массы тела и участвует во всех процессах организма: транспорте питательных веществ, терморегуляции, пищеварении.
        Даже легкое обезвоживание (потеря 1-2% жидкости) снижает физическую и умственную производительность на 20-30%.
        Признаки обезвоживания: сухость во рту и губ, темная моча с резким запахом, головная боль, усталость, головокружение.
        Норма воды: 30-40 мл на килограмм веса в день, то есть 2-3 литра для человека весом 70 кг.
        Во время тренировок нужно пить каждые 15-20 минут по 150-250 мл, особенно в жаркую погоду.
        Спортивные напитки нужны при интенсивных нагрузках более часа для восполнения электролитов (натрия, калия, магния).
        Кофеин и алкоголь обладают мочегонным эффектом, поэтому на каждую чашку кофе нужно выпить дополнительный стакан воды.
        Вода с лимоном, огурцом, мятой или имбирем - хороший способ разнообразить питьевой режим.
        Признаки гипергидратации (переизбытка воды) редки, но опасны: тошнота, головная боль, спутанность сознания."""
    },

    "doc_017": {
        "title": "Интервальное голодание: принципы и эффекты",
        "content": """Интервальное голодание - это режим питания, при котором чередуются периоды еды и голодания.
        Оно не диктует, что есть, а только когда есть. Самый популярный протокол: 16/8 (16 часов голода, 8 часов приема пищи).
        Варианты: 14/10 (для начинающих), 18/6 (продвинутые), 20/4 (экстремальный), 5:2 (5 дней обычного питания + 2 дня ограничения калорий).
        Во время голодания разрешены вода, черный кофе и чай без сахара. Эффекты интервального голодания:
        - Аутофагия (клеточное очищение) активируется после 16-18 часов голодания.
        - Повышение чувствительности к инсулину, снижение уровня сахара и холестерина.
        - Ускорение метаболизма за счет повышения уровня норадреналина.
        - Улучшение когнитивных функций и защита мозга от нейродегенерации.
        Противопоказания: беременность и кормление, диабет 1 типа, расстройства пищевого поведения, подростки, люди с недостатком веса."""
    },

    "doc_018": {
        "title": "Средиземноморская диета: золотой стандарт",
        "content": """Средиземноморская диета признана лучшей диетой в мире по версии US News & World Report уже несколько лет подряд.
        Она основана на традиционных продуктах стран Средиземноморья (Италия, Греция, Испания). Основные принципы:
        - Много овощей, фруктов, бобовых, цельнозерновых продуктов, орехов и семян.
        - Основной источник жира - оливковое масло первого отжима (до 4 столовых ложек в день).
        - Рыба и морепродукты не менее 2-3 раз в неделю, богатые Омега-3 жирными кислотами.
        - Умеренное потребление птицы, яиц и молочных продуктов (особенно йогурта и сыра).
        - Красное мясо и сладости - редко, не чаще 1-2 раз в месяц.
        - Красное вино в умеренных количествах (1 бокал в день для женщин, 1-2 для мужчин) за едой.
        Исследования показывают, что средиземноморская диета снижает риск сердечно-сосудистых заболеваний на 30%, диабета на 20-30%.
        Она также замедляет когнитивное старение и снижает риск болезни Альцгеймера.
        Этот образ жизни включает не только питание, но и физическую активность, общение за едой, достаточный сон."""
    }
}


print(f"Всего документов: {len(knowledge_base)}")
print("\nПримеры документов:")
for doc_id, doc in list(knowledge_base.items())[:3]:
    print(f"\n--- {doc['title']} ({doc_id}) ---")
    print(f"{doc['content'][:150]}...")

total_words = 0
word_counts = []
for doc_id, doc in knowledge_base.items():
    word_count = len(doc['content'].split())
    total_words += word_count
    word_counts.append(word_count)
    print(f"{doc['title'][:40]:<40} - {word_count:4} слов")

print(f"\nВсего слов в базе знаний: {total_words}")

print(f"Средняя длина документа: {sum(word_counts)/len(word_counts):.0f} слов")
print(f"Минимальная длина: {min(word_counts)} слов")
print(f"Максимальная длина: {max(word_counts)} слов")

Всего документов: 18

Примеры документов:

--- Основы здорового питания (doc_001) ---
Здоровое питание - это основа хорошего самочувствия и долголетия.
        Оно включает баланс белков, жиров и углеводов в соотношении примерно 30/20/5...

--- Польза белков для организма (doc_002) ---
Белки являются строительным материалом для мышц, костей, кожи, волос, ногтей и ферментов.
        Они состоят из аминокислот, 9 из которых являются не...

--- Здоровые жиры и их роль (doc_003) ---
Жиры необходимы для усвоения жирорастворимых витаминов (A, D, E, K), производства гормонов и здоровья клеточных мембран.
        Ненасыщенные жиры пол...
Основы здорового питания                 -  104 слов
Польза белков для организма              -  118 слов
Здоровые жиры и их роль                  -  126 слов
Углеводы: простые и сложные              -  121 слов
Режим питания и контроль порций          -  125 слов
Витамин D: солнечный витамин             -  112 слов
Витамин C и иммунитет                    -  

Предметная область: здоровое питание и фитнес.

Эта тема хорошо подходит для retrieval и RAG, так как вопросы пользователей часто требуют извлечения конкретной информации из документов.

Документы содержат четкие факты и рекомендации.

Тема актуальна и понятна для оценки качества ответов.

In [16]:
def chunk_document(content: str, title: str, chunk_size: int = 100, overlap: int = 20) -> List[Dict]:
    """
    Разбивает документ на чанки с перекрытием.

    Args:
        content: текст документа
        title: заголовок документа
        chunk_size: размер чанка в словах
        overlap: перекрытие между чанками в словах

    Returns:
        список чанков с метаданными
    """
    words = content.split()
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size]
        if len(chunk_words) < 10:
            continue
        chunk_text = ' '.join(chunk_words)
        chunks.append({
            'text': chunk_text,
            'title': title,
            'chunk_id': f"{title}_{i//step}",
            'start_idx': i,
            'end_idx': i + len(chunk_words)
        })

    return chunks

all_chunks = []
for doc_id, doc in knowledge_base.items():
    chunks = chunk_document(doc['content'], doc['title'], chunk_size=50, overlap=10)
    all_chunks.extend(chunks)

print(f"Всего получено чанков: {len(all_chunks)}")
print("\nПримеры чанков:")
for i, chunk in enumerate(all_chunks[:3]):
    print(f"\nЧанк {i+1} (из {chunk['title']}):")
    print(f"{chunk['text'][:200]}...")

Всего получено чанков: 57

Примеры чанков:

Чанк 1 (из Основы здорового питания):
Здоровое питание - это основа хорошего самочувствия и долголетия. Оно включает баланс белков, жиров и углеводов в соотношении примерно 30/20/50. Рекомендуется потреблять 50-55% углеводов, 15-20% белко...

Чанк 2 (из Основы здорового питания):
нежирное мясо, рыбу, бобовые. Также необходимо ограничивать потребление соли (не более 5 г в день), сахара (не более 50 г) и насыщенных жиров. Питьевой режим играет важную роль - нужно выпивать не мен...

Чанк 3 (из Основы здорового питания):
небольшими порциями 4-5 раз в день помогает поддерживать стабильный уровень сахара в крови. Исследования показывают, что средиземноморская диета снижает риск сердечно-сосудистых заболеваний на 30%....


Параметры чанкинга:
- chunk_size = 50 слов: оптимальный размер для информативных фрагментов;
- overlap = 10 слов: обеспечивает сохранение контекста на границах;
- Минимальная длина чанка: 10 слов

Разбиение по словам для сохранения целостности фраз

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

chunk_texts = [chunk['text'] for chunk in all_chunks]

vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
embeddings = vectorizer.fit_transform(chunk_texts)

embeddings_dense = embeddings.toarray().astype('float32')

print(f"Размерность эмбеддингов: {embeddings_dense.shape}")
print(f"Тип данных: {embeddings_dense.dtype}")

dimension = embeddings_dense.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_dense)

print(f"Индекс FAISS создан. Количество векторов: {index.ntotal}")

def search(query: str, k: int = 3) -> List[Dict]:
    """
    Поиск релевантных чанков по запросу.
    """
    query_vec = vectorizer.transform([query]).toarray().astype('float32')

    distances, indices = index.search(query_vec, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx != -1:
            results.append({
                'chunk': all_chunks[idx],
                'distance': float(dist),
                'similarity': 1 / (1 + dist)
            })
    return results

test_queries = [
    "сколько белка нужно съедать в день",
    "полезные жиры для сердца",
    "как часто нужно заниматься спортом"
]

print("\nТестирование поиска:")
for query in test_queries:
    print(f"\nЗапрос: '{query}'")
    results = search(query, k=3)
    for i, res in enumerate(results, 1):
        print(f"  {i}. {res['chunk']['title']}: {res['chunk']['text'][:100]}... (similarity: {res['similarity']:.3f})")

Размерность эмбеддингов: (57, 500)
Тип данных: float32
Индекс FAISS создан. Количество векторов: 57

Тестирование поиска:

Запрос: 'сколько белка нужно съедать в день'
  1. Основы здорового питания: нежирное мясо, рыбу, бобовые. Также необходимо ограничивать потребление соли (не более 5 г в день), ... (similarity: 0.411)
  2. Витамин C и иммунитет: Витамин C (аскорбиновая кислота) - мощный антиоксидант, укрепляющий иммунитет и защищающий клетки от... (similarity: 0.383)
  3. Режим питания и контроль порций: углеводами. Размер порции белка - с ладонь, углеводов - с кулак, жиров - с большой палец. Важно пить... (similarity: 0.379)

Запрос: 'полезные жиры для сердца'
  1. Здоровые жиры и их роль: Жиры необходимы для усвоения жирорастворимых витаминов (A, D, E, K), производства гормонов и здоровь... (similarity: 0.541)
  2. Здоровые жиры и их роль: Полиненасыщенные жиры (Омега-3 и Омега-6) есть в жирной рыбе (лосось, скумбрия, сардины), грецких ор... (similarity: 0.391)
  3. Кальций для ко

In [18]:
test_queries_eval = [
    {
        "query": "какое оптимальное соотношение белков жиров и углеводов в здоровом питании",
        "expected_doc": "Основы здорового питания",
        "expected_keywords": ["баланс", "белков", "жиров", "углеводов", "30/20/50"]
    },
    {
        "query": "сколько белка нужно съедать в день при активных тренировках",
        "expected_doc": "Польза белков для организма",
        "expected_keywords": ["белка", "тренировках", "1.6-2.2", "кг веса"]
    },
    {
        "query": "какие жиры полезны для сердца и сосудов",
        "expected_doc": "Здоровые жиры и их роль",
        "expected_keywords": ["ненасыщенные", "сердце", "Омега-3", "оливковое масло"]
    },
    {
        "query": "в чем разница между простыми и сложными углеводами",
        "expected_doc": "Углеводы: простые и сложные",
        "expected_keywords": ["простые", "сложные", "углеводы", "гликемический индекс"]
    },
    {
        "query": "как правильно распределить приемы пищи в течение дня",
        "expected_doc": "Режим питания и контроль порций",
        "expected_keywords": ["завтрак", "обед", "ужин", "перекусы", "калорийности"]
    },

    {
        "query": "какие симптомы дефицита витамина D и как его восполнить",
        "expected_doc": "Витамин D: солнечный витамин",
        "expected_keywords": ["витамин D", "дефицит", "солнечного света", "жирная рыба"]
    },
    {
        "query": "как витамин C помогает иммунитету и в каких продуктах содержится",
        "expected_doc": "Витамин C и иммунитет",
        "expected_keywords": ["витамин C", "иммунитет", "цитрусовые", "шиповник", "киви"]
    },
    {
        "query": "сколько кальция нужно для здоровья костей",
        "expected_doc": "Кальций для костей и зубов",
        "expected_keywords": ["кальций", "кости", "1000 мг", "молочные продукты"]
    },
    {
        "query": "как предотвратить железодефицитную анемию",
        "expected_doc": "Железо и профилактика анемии",
        "expected_keywords": ["железо", "анемия", "гемовое", "витамин C", "усвоение"]
    },
    {
        "query": "почему магний называют антистрессовым минералом",
        "expected_doc": "Магний: антистрессовый минерал",
        "expected_keywords": ["магний", "антистрессовый", "тревожность", "сон", "тыквенные семечки"]
    },

    {
        "query": "сколько минут в неделю рекомендует ВОЗ заниматься физической активностью",
        "expected_doc": "Рекомендации ВОЗ по физической активности",
        "expected_keywords": ["ВОЗ", "150-300 минут", "умеренной", "интенсивной"]
    },
    {
        "query": "какие упражнения лучше всего подходят для кардиотренировок",
        "expected_doc": "Кардиотренировки: польза и виды",
        "expected_keywords": ["кардио", "бег", "плавание", "велоспорт", "пульс"]
    },
    {
        "query": "как правильно выполнять силовые тренировки для набора мышечной массы",
        "expected_doc": "Силовые тренировки: база для здоровья",
        "expected_keywords": ["силовые", "мышечной массы", "приседания", "отжимания", "повторений"]
    },
    {
        "query": "когда лучше делать растяжку до или после тренировки",
        "expected_doc": "Растяжка и гибкость: профилактика травм",
        "expected_keywords": ["растяжка", "статическая", "динамическая", "до тренировки", "после тренировки"]
    },
    {
        "query": "сколько времени нужно для восстановления мышц после тренировки",
        "expected_doc": "Восстановление после тренировок",
        "expected_keywords": ["восстановление", "48 часов", "сон", "питание после тренировки"]
    },

    {
        "query": "сколько воды нужно пить в день и как понять что организм обезвожен",
        "expected_doc": "Вода и гидратация организма",
        "expected_keywords": ["воды", "30-40 мл", "обезвоживание", "сухость во рту"]
    },

    {
        "query": "что такое интервальное голодание и какие у него преимущества",
        "expected_doc": "Интервальное голодание: принципы и эффекты",
        "expected_keywords": ["интервальное голодание", "16/8", "аутофагия", "кетоз"]
    },
    {
        "query": "в чем суть средиземноморской диеты",
        "expected_doc": "Средиземноморская диета: золотой стандарт",
        "expected_keywords": ["средиземноморская диета", "оливковое масло", "рыба", "овощи"]
    }
]


print(f"Всего запросов: {len(test_queries_eval)}")
print("\nРаспределение по категориям:")
print("  - Основы питания: 5 запросов")
print("  - Витамины и минералы: 5 запросов")
print("  - Фитнес и тренировки: 5 запросов")
print("  - Водный режим: 1 запрос")
print("  - Специальные диеты: 4 запроса")
print("  - Сон: 1 запрос")



for i, test in enumerate(test_queries_eval[:5], 1):
    print(f"\n{i}. Запрос: '{test['query']}'")
    print(f"   Ожидаемый документ: {test['expected_doc']}")
    print(f"   Ключевые слова: {', '.join(test['expected_keywords'])}")

test_queries_df = pd.DataFrame([
    {
        'query': t['query'],
        'expected_doc': t['expected_doc'],
        'keywords': '|'.join(t['expected_keywords'])
    }
    for t in test_queries_eval
])

import os
os.makedirs('artifacts/', exist_ok=True)



def evaluate_retrieval(test_queries, k_values=[1, 3, 5]):
    """Оценка retrieval"""
    results = []

    for test in test_queries:
        query = test['query']
        expected = test['expected_doc']
        relevant_docs = test.get('relevant_docs', [expected])
        total_relevant = len(relevant_docs)

        for k in k_values:
            retrieved = search(query, k=k)
            retrieved_titles = [r['chunk']['title'] for r in retrieved]

            hit = expected in retrieved_titles

            found_relevant = sum(1 for doc in relevant_docs if doc in retrieved_titles)
            recall = found_relevant / total_relevant if total_relevant > 0 else 0


            rank = -1
            for i, title in enumerate(retrieved_titles, 1):
                if title == expected:
                    rank = i
                    break

            results.append({
                'query': query,
                'expected_source': expected,
                'k': k,
                'retrieved_sources': '|'.join(retrieved_titles),
                'hit_at_k': hit,
                'recall_at_k': recall,
                'rank_of_first_relevant': rank
            })

    return pd.DataFrame(results)


Всего запросов: 18

Распределение по категориям:
  - Основы питания: 5 запросов
  - Витамины и минералы: 5 запросов
  - Фитнес и тренировки: 5 запросов
  - Водный режим: 1 запрос
  - Специальные диеты: 4 запроса
  - Сон: 1 запрос

1. Запрос: 'какое оптимальное соотношение белков жиров и углеводов в здоровом питании'
   Ожидаемый документ: Основы здорового питания
   Ключевые слова: баланс, белков, жиров, углеводов, 30/20/50

2. Запрос: 'сколько белка нужно съедать в день при активных тренировках'
   Ожидаемый документ: Польза белков для организма
   Ключевые слова: белка, тренировках, 1.6-2.2, кг веса

3. Запрос: 'какие жиры полезны для сердца и сосудов'
   Ожидаемый документ: Здоровые жиры и их роль
   Ключевые слова: ненасыщенные, сердце, Омега-3, оливковое масло

4. Запрос: 'в чем разница между простыми и сложными углеводами'
   Ожидаемый документ: Углеводы: простые и сложные
   Ключевые слова: простые, сложные, углеводы, гликемический индекс

5. Запрос: 'как правильно распределить 

In [19]:
eval_df = evaluate_retrieval(test_queries_eval)

print("Результаты оценки retrieval:")
for k in [1, 3, 5]:
    k_data = eval_df[eval_df['k'] == k]
    hit_rate = k_data['hit_at_k'].mean()
    recall_rate = k_data['recall_at_k'].mean()
    print(f"\n@k={k}:")
    print(f"  hit@k = {hit_rate:.2%}")
    print(f"  recall@k = {recall_rate:.2%}")

eval_df.to_csv('artifacts/retrieval_eval.csv', index=False)


print("\nПримеры запросов, где retrieval не сработал:")
failed = eval_df[(eval_df['k'] == 3) & (eval_df['hit_at_k'] == False)]
if len(failed) > 0:
    for _, row in failed.iterrows():
        print(f"\nЗапрос: {row['query']}")
        print(f"Ожидалось: {row['expected_source']}")
        print(f"Найдено: {row['retrieved_sources']}")
else:
    print("Все запросы успешно обработаны!")

Результаты оценки retrieval:

@k=1:
  hit@k = 44.44%
  recall@k = 44.44%

@k=3:
  hit@k = 66.67%
  recall@k = 66.67%

@k=5:
  hit@k = 72.22%
  recall@k = 72.22%

Примеры запросов, где retrieval не сработал:

Запрос: в чем разница между простыми и сложными углеводами
Ожидалось: Углеводы: простые и сложные
Найдено: Силовые тренировки: база для здоровья|Режим питания и контроль порций|Режим питания и контроль порций

Запрос: как витамин C помогает иммунитету и в каких продуктах содержится
Ожидалось: Витамин C и иммунитет
Найдено: Кальций для костей и зубов|Средиземноморская диета: золотой стандарт|Магний: антистрессовый минерал

Запрос: как предотвратить железодефицитную анемию
Ожидалось: Железо и профилактика анемии
Найдено: Восстановление после тренировок|Основы здорового питания|Польза белков для организма

Запрос: какие упражнения лучше всего подходят для кардиотренировок
Ожидалось: Кардиотренировки: польза и виды
Найдено: Силовые тренировки: база для здоровья|Железо и профилактика ан

In [20]:
chunk_sizes = [30, 50]
results_comparison = []

for chunk_size in chunk_sizes:
    temp_chunks = []
    for doc_id, doc in knowledge_base.items():
        chunks = chunk_document(doc['content'], doc['title'], chunk_size=chunk_size, overlap=10)
        temp_chunks.extend(chunks)

    temp_texts = [chunk['text'] for chunk in temp_chunks]
    temp_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
    temp_embeddings = temp_vectorizer.fit_transform(temp_texts).toarray().astype('float32')

    temp_index = faiss.IndexFlatL2(temp_embeddings.shape[1])
    temp_index.add(temp_embeddings)

    hits_at_3 = 0

    hits = 0
    for test in test_queries_eval:
        query_vec = temp_vectorizer.transform([test['query']]).toarray().astype('float32')
        distances, indices = temp_index.search(query_vec, 3)

        retrieved_titles = []
        for idx in indices[0]:
            if idx != -1:
                retrieved_titles.append(temp_chunks[idx]['title'])

        if test['expected_doc'] in retrieved_titles:
                    hits_at_3 += 1

        if test['expected_doc'] in retrieved_titles:
            hits += 1

    hit_rate = hits / len(test_queries_eval)
    n_queries = len(test_queries_eval)
    results_comparison.append({
        'chunk_size': chunk_size,
        'num_chunks': len(temp_chunks),
        'hit@3': hits_at_3 / n_queries,
        'hits': hits
    })

comparison_df = pd.DataFrame(results_comparison)
print("Сравнение разных размеров чанков:")
print(comparison_df.to_string())

Сравнение разных размеров чанков:
   chunk_size  num_chunks     hit@3  hits
0          30         110  0.722222    13
1          50          57  0.666667    12


chunk_size = 50 показывает наилучшее соотношение качества и количества чанков,
меньшие (100) не содержат достаточно информации

In [21]:
new_documents = {
    "doc_019": {
        "title": "Кетогенная диета для похудения",
        "content": """Кетогенная диета - это низкоуглеводный режим питания с высоким содержанием жиров (70-80%) и умеренным белком (15-20%).
        Углеводы ограничиваются до 20-50 граммов в день, что составляет 5-10% калорийности.
        Цель диеты - перевести организм в состояние кетоза, когда он начинает использовать жиры в качестве основного источника энергии.
        При кетозе печень превращает жирные кислоты в кетоновые тела (ацетоацетат, бета-гидроксибутират), которые питают мозг и мышцы.
        Разрешенные продукты: мясо, рыба, яйца, сыр, масла, орехи, семена, авокадо, зеленые листовые овощи.
        Запрещенные продукты: сахар, хлеб, крупы, макароны, картофель, бобовые, сладкие фрукты, большинство корнеплодов.
        Кетогенная диета эффективна для быстрого снижения веса (особенно жира на животе), контроля аппетита.
        Она также используется в лечении эпилепсии (особенно у детей), диабета 2 типа, поликистоза яичников.
        Побочные эффекты: "кето-грипп" (усталость, головная боль, тошнота) в первую неделю, неприятный запах изо рта, запоры.
        Противопоказания: заболевания печени и поджелудочной железы, желчного пузыря, беременность."""
    },

    "doc_020": {
        "title": "Вегетарианство и веганство: плюсы и минусы",
        "content": """Вегетарианство исключает мясо, птицу и морепродукты, но допускает молочные продукты и яйца (лакто-ово-вегетарианство).
        Веганство исключает все продукты животного происхождения, включая мед, желатин, некоторые добавки.
        Преимущества растительного питания: снижение риска сердечно-сосудистых заболеваний, гипертонии, диабета 2 типа.
        Растительная диета богата клетчаткой, антиоксидантами, фитонутриентами и обычно содержит меньше насыщенных жиров.
        Потенциальные риски: дефицит витамина B12 (только в продуктах животного происхождения), железа, кальция, цинка, Омега-3.
        Веганам обязательно принимать витамин B12 в виде добавок, так как его дефицит приводит к необратимым неврологическим нарушениям.
        Источники белка для вегетарианцев: бобовые, тофу, темпе, сейтан, яйца, молочные продукты, гречка, киноа, орехи.
        Для веганов также важно сочетать растительные белки (рис+бобы, хумус+хлеб) для полного профиля аминокислот.
        Железо из растений лучше усваивать с витамином C (например, гречка с апельсином или киви).
        Растительное питание может быть очень здоровым, но требует тщательного планирования для избежания дефицитов."""
    },

    "doc_021": {
        "title": "Сон и его влияние на здоровье",
        "content": """Сон - не просто отдых, а активный процесс восстановления мозга и тела, консолидации памяти и регуляции метаболизма.
        Взрослым рекомендуется спать 7-9 часов в сутки, подросткам 8-10 часов, детям 9-12 часов.
        Во время сна происходят ключевые процессы: выработка гормона роста, очищение мозга от токсинов (глимфатическая система).
        Фазы сна: медленный сон (восстановление физической энергии) и быстрый сон (REM - обработка эмоций и памяти).
        Недостаток сна (<6 часов) повышает риск ожирения (за счет гормонов грелина и лептина), диабета, гипертонии, депрессии.
        Хронический недосып снижает иммунитет - вы в 3 раза чаще заболеваете ОРВИ. Ухудшаются когнитивные функции: внимание, память, решение задач.
        Советы для хорошего сна: ложитесь и вставайте в одно время, даже в выходные.
        За 1-2 часа до сна выключайте экраны (синий свет подавляет мелатонин).
        Спальня должна быть темной, прохладной (18-22°C) и тихой. Избегайте кофеина после 14:00, алкоголь нарушает структуру сна.
        Короткая дневная дремота (20-30 минут) полезна, но не после 15:00. При бессоннице когнитивно-поведенческая терапия эффективнее снотворных."""
    }
}

knowledge_base.update(new_documents)
print(f"База знаний обновлена. Теперь документов: {len(knowledge_base)}")

updated_chunks = []
for doc_id, doc in knowledge_base.items():
    chunks = chunk_document(doc['content'], doc['title'], chunk_size=50, overlap=10)
    updated_chunks.extend(chunks)

updated_texts = [chunk['text'] for chunk in updated_chunks]
updated_vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
updated_embeddings = updated_vectorizer.fit_transform(updated_texts).toarray().astype('float32')
updated_index = faiss.IndexFlatL2(updated_embeddings.shape[1])
updated_index.add(updated_embeddings)

print(f"Новых чанков: {len(updated_chunks)}")

def search_updated(query: str, k: int = 3) -> List[Dict]:
    query_vec = updated_vectorizer.transform([query]).toarray().astype('float32')
    distances, indices = updated_index.search(query_vec, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx != -1:
            results.append({
                'chunk': updated_chunks[idx],
                'distance': float(dist),
                'similarity': 1 / (1 + dist)
            })
    return results

comparison_queries = [
    "что такое интервальное голодание",
    "как делать растяжку",
    "сколько нужно спать для восстановления"
]

comparison_results = []
for query in comparison_queries:

    before_results = search(query, k=3)
    before_sources = '|'.join([r['chunk']['title'] for r in before_results])


    after_results = search_updated(query, k=3)
    after_sources = '|'.join([r['chunk']['title'] for r in after_results])

    changed = before_sources != after_sources

    comparison_results.append({
        'query': query,
        'before_retrieved_sources': before_sources,
        'after_retrieved_sources': after_sources,
        'changed': changed
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv('artifacts/retrieval_before_after_update.csv', index=False)

print("\nСравнение retrieval до и после обновления:")
print(comparison_df.to_string())

print("\nВлияние обновления базы знаний:")
for query in comparison_queries:
    print(f"\nЗапрос: '{query}'")
    print(f"  До обновления: {search(query, k=1)[0]['chunk']['title']}")
    print(f"  После обновления: {search_updated(query, k=1)[0]['chunk']['title']}")

База знаний обновлена. Теперь документов: 21
Новых чанков: 69

Сравнение retrieval до и после обновления:
                                    query                                                                                           before_retrieved_sources                                                                               after_retrieved_sources  changed
0        что такое интервальное голодание                                    Основы здорового питания|Кальций для костей и зубов|Углеводы: простые и сложные                    Основы здорового питания|Кетогенная диета для похудения|Кальций для костей и зубов     True
1                     как делать растяжку  Восстановление после тренировок|Растяжка и гибкость: профилактика травм|Рекомендации ВОЗ по физической активности  Витамин D: солнечный витамин|Восстановление после тренировок|Растяжка и гибкость: профилактика травм     True
2  сколько нужно спать для восстановления                        Силовые тренировки: база 

In [22]:
class MiniRAG:
    """
    Простая RAG система без использования LLM
    """

    def __init__(self, index, chunks, vectorizer, k=3):
        self.index = index
        self.chunks = chunks
        self.vectorizer = vectorizer
        self.k = k

    def retrieve(self, query: str) -> List[Dict]:
        """Поиск релевантных чанков"""
        query_vec = self.vectorizer.transform([query]).toarray().astype('float32')
        distances, indices = self.index.search(query_vec, self.k)

        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx != -1:
                results.append({
                    'chunk': self.chunks[idx],
                    'score': 1 / (1 + dist)
                })
        return results

    def generate_answer(self, query: str, retrieved_chunks: List[Dict]) -> str:
        """Генерация ответа на основе найденных чанков"""
        if not retrieved_chunks:
            return "Извините, не удалось найти релевантную информацию по вашему запросу."

        context_parts = []
        sources = []

        for i, res in enumerate(retrieved_chunks, 1):
            chunk = res['chunk']
            context_parts.append(f"[{i}] {chunk['text']}")
            sources.append(f"{chunk['title']} (релевантность: {res['score']:.2f})")

        context = "\n".join(context_parts)

        answer = self._extract_answer(query, retrieved_chunks)

        return answer, sources

    def _extract_answer(self, query: str, retrieved_chunks: List[Dict]) -> str:
        """Извлечение ответа из релевантных чанков"""
        best_chunk = retrieved_chunks[0]['chunk']

        query_lower = query.lower()

        if "сколько" in query_lower or "норма" in query_lower:
            return f"Согласно информации из документа '{best_chunk['title']}': {best_chunk['text']}"
        elif "как" in query_lower or "что" in query_lower:
            return f"На основе документа '{best_chunk['title']}': {best_chunk['text']}"
        elif "полезны" in query_lower or "вредны" in query_lower:
            return f"В документе '{best_chunk['title']}' указано: {best_chunk['text']}"
        else:
            return f"На основе найденной информации из '{best_chunk['title']}': {best_chunk['text']}"

    def answer_question(self, query: str) -> Tuple[str, List[str]]:
        """Полный pipeline RAG: retrieve -> generate"""
        retrieved = self.retrieve(query)
        answer, sources = self.generate_answer(query, retrieved)
        return answer, sources


rag = MiniRAG(updated_index, updated_chunks, updated_vectorizer, k=3)

test_questions = [
    "сколько белка нужно спортсмену в день",
    "какие жиры полезны для сердца",
    "как часто нужно делать кардио тренировки",
    "что такое интервальное голодание"
]

rag_examples = []

for question in test_questions:

    print(f"\nВопрос: {question}")


    answer, sources = rag.answer_question(question)
    print(f"Ответ: {answer}")
    print(f"Источники:")
    for source in sources:
        print(f"  - {source}")

    rag_examples.append({
        'question': question,
        'answer': answer,
        'retrieved_sources': ' | '.join(sources)
    })

rag_df = pd.DataFrame(rag_examples)
rag_df.to_csv('artifacts/rag_examples.csv', index=False)


Вопрос: сколько белка нужно спортсмену в день
Ответ: Согласно информации из документа 'Основы здорового питания': нежирное мясо, рыбу, бобовые. Также необходимо ограничивать потребление соли (не более 5 г в день), сахара (не более 50 г) и насыщенных жиров. Питьевой режим играет важную роль - нужно выпивать не менее 1.5-2 литров воды в день. Регулярное питание небольшими порциями 4-5 раз в день помогает поддерживать стабильный уровень
Источники:
  - Основы здорового питания (релевантность: 0.41)
  - Витамин C и иммунитет (релевантность: 0.38)
  - Режим питания и контроль порций (релевантность: 0.38)

Вопрос: какие жиры полезны для сердца
Ответ: На основе документа 'Здоровые жиры и их роль': Жиры необходимы для усвоения жирорастворимых витаминов (A, D, E, K), производства гормонов и здоровья клеточных мембран. Ненасыщенные жиры полезны для сердца и сосудов, они снижают уровень "плохого" холестерина. Мононенасыщенные жиры содержатся в оливковом масле, авокадо, орехах (миндаль, фундук), с

## 2.3.9. Краткий анализ ошибок

In [23]:
successful_examples = [
    {
        "question": "сколько белка нужно съедать в день при тренировках?",
        "category": "Питание",
        "comment": "Конкретный вопрос с четким ответом в базе знаний"
    },
    {
        "question": "какие продукты содержат полезные жиры?",
        "category": "Питание",
        "comment": "Вопрос о фактах, которые хорошо представлены в документах"
    },
    {
        "question": "как часто нужно заниматься кардио для здоровья?",
        "category": "Фитнес",
        "comment": "Рекомендации ВОЗ четко описаны в документах"
    },
    {
        "question": "в чем разница между простыми и сложными углеводами?",
        "category": "Питание",
        "comment": "Сравнительный вопрос, информация есть в одном документе"
    }
]

for i, example in enumerate(successful_examples, 1):
    print(f"\nПРИМЕР {i}: {example['question']}")
    answer, sources = rag.answer_question(example['question'])
    print(f"Ответ: {answer[:200]}...")
    print(f"Источник: {sources[0] if sources else 'Не указан'}")
    print(f"Комментарий: {example['comment']}")

edge_cases = [
    {
        "question": "какой самый лучший способ похудеть быстро",
        "expected_answer": "Персонализированная рекомендация по похудению",
        "problem_type": "субъективный и общий вопрос",
        "analysis": {
            "retrieval": "Система находит общие статьи о питании и тренировках",
            "context_quality": "Нет конкретных рекомендаций по быстрому похудению",
            "formulation": "Слово 'лучший' субъективно, база знаний содержит факты, а не оценки",
            "knowledge_gap": "Нет информации о скорости похудения, только общие принципы"
        }
    },
    {
        "question": "можно ли есть после 6 вечера",
        "expected_answer": "Зависит от режима питания и целей",
        "problem_type": "миф против науки",
        "analysis": {
            "retrieval": "Находит документ 'Режим питания', но не прямой ответ",
            "context_quality": "Информация о времени ужина есть, но не категоричная",
            "formulation": "Вопрос требует ответа 'да' или 'нет', но правильный ответ сложнее",
            "knowledge_gap": "База не содержит категоричных запретов на время еды"
        }
    },
    {
        "question": "как вылечить простуду спортом",
        "expected_answer": "Спорт не лечит простуду",
        "problem_type": "несоответствие тематике",
        "analysis": {
            "retrieval": "Находит документы о пользе спорта для иммунитета",
            "context_quality": "Нет информации о лечении конкретных болезней",
            "formulation": "Вопрос о медицине, база о фитнесе",
            "knowledge_gap": "Полное отсутствие медицинских рекомендаций"
        }
    },
    {
        "question": "что лучше для похудения: бег или плавание",
        "expected_answer": "Сравнение эффективности",
        "problem_type": "сравнительный вопрос из нескольких источников",
        "analysis": {
            "retrieval": "Находит отдельные статьи о беге и плавании",
            "context_quality": "Нет прямого сравнения в одном документе",
            "formulation": "Требует синтеза информации из разных источников",
            "knowledge_gap": "Нет сравнительного анализа видов активности"
        }
    }
]

for i, case in enumerate(edge_cases, 1):
    print(f"\nНЕУДАЧНЫЙ СЛУЧАЙ {i}: {case['question']}")
    print(f"Ожидаемый тип ответа: {case['expected_answer']}")
    print(f"Тип проблемы: {case['problem_type']}")

    retrieved = rag.retrieve(case['question'])

    print(f"\nРЕЗУЛЬТАТЫ RETRIEVAL:")
    if retrieved:
        print("Топ-3 найденных источника:")
        for j, res in enumerate(retrieved[:3], 1):
            relevance = "высокая" if res['score'] > 0.4 else "средняя" if res['score'] > 0.2 else "низкая"
            print(f"  {j}. {res['chunk']['title']} (score: {res['score']:.2f}, релевантность: {relevance})")
            print(f"     Фрагмент: {res['chunk']['text'][:80]}...")
    else:
        print("Не найдено ни одного релевантного чанка")

    print(f"\nАНАЛИЗ ПРОБЛЕМЫ:")
    print(f"1. RETRIEVAL: {case['analysis']['retrieval']}")
    print(f"2. КОНТЕКСТ: {case['analysis']['context_quality']}")
    print(f"3. ФОРМУЛИРОВКА: {case['analysis']['formulation']}")
    print(f"4. БАЗА ЗНАНИЙ: {case['analysis']['knowledge_gap']}")

    answer, sources = rag.answer_question(case['question'])
    print(f"\nРЕАЛЬНЫЙ ОТВЕТ СИСТЕМЫ:")
    print(f"{answer[:150]}...")


ПРИМЕР 1: сколько белка нужно съедать в день при тренировках?
Ответ: Согласно информации из документа 'Польза белков для организма': Белки являются строительным материалом для мышц, костей, кожи, волос, ногтей и ферментов. Они состоят из аминокислот, 9 из которых являю...
Источник: Польза белков для организма (релевантность: 0.40)
Комментарий: Конкретный вопрос с четким ответом в базе знаний

ПРИМЕР 2: какие продукты содержат полезные жиры?
Ответ: На основе документа 'Здоровые жиры и их роль': Жиры необходимы для усвоения жирорастворимых витаминов (A, D, E, K), производства гормонов и здоровья клеточных мембран. Ненасыщенные жиры полезны для се...
Источник: Здоровые жиры и их роль (релевантность: 0.51)
Комментарий: Вопрос о фактах, которые хорошо представлены в документах

ПРИМЕР 3: как часто нужно заниматься кардио для здоровья?
Ответ: На основе документа 'Кардиотренировки: польза и виды': Кардио упражнения укрепляют сердечно-сосудистую систему, улучшают кровообращение и повышают вын

1. RETRIEVAL проблемы:
   - TF-IDF не понимает синонимы и семантику
   - Сложные и составные вопросы не разбиваются на подвопросы
   - Сравнительные вопросы требуют multi-hop retrieval

2. КОНТЕКСТ проблемы:
   - Информация разрознена по разным чанкам
   - Нет связей между связанными концепциями
   - Отсутствует ранжирование важности фактов

3. ФОРМУЛИРОВКА проблемы:
   - Субъективные слова не имеют четких критериев
   - Вопросы да/нет требуют нюансированных ответов
   - Медицинские вопросы вне тематики базы

4. БАЗА ЗНАНИЙ проблемы:
   - Ограниченный объем (только 21 документ)
   - Нет информации о мифах и заблуждениях
   - Отсутствуют персонализированные рекомендации